<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/unknown_jobs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/unknown_jobs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Job Category Chatbot · MIST 4600 Honors Option

My app helps **job boards** decide **where to repair missing categories and which sector campaigns to test** by comparing job postings across eight countries.

1. **Q1:** Which countries have the most missing categories?
2. **Q2:** How often did Gemini's labels match existing sectors, and what types were hidden?
3. **Q3 (my own idea):** How did the leading sectors change after filling missing categories?

The notebook uses the full 4,000-row dataset. Q2 and Q3 use the completed aggregate results of an earlier 4,000-title Gemini run; the live example labels one more title so you can see the outside tool work. The aggregate is a saved result, not a new 4,000-title classification each time you run the notebook.

## Step 1 · Load the two public CSV files

The first CSV has all 4,000 original postings. The second has completed aggregate counts from the earlier Gemini run. Both are in this GitHub repository, so **Runtime → Run all** does not need Drive. For the live Gemini example, save `GOOGLE_API_KEY` in Colab Secrets and turn on notebook access. No key is typed into the code.

In [ ]:
!pip -q install google-genai
import pandas as pd
from google.colab import userdata
from google import genai

BASE = "https://raw.githubusercontent.com/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/main/"
jobs = pd.read_csv(BASE + "dfall_clean.csv")
completed = pd.read_csv(BASE + "completed_results.csv")
print("Postings loaded:", len(jobs))

Postings loaded: 4000


## Step 2 · Answer the three questions with pandas

Q1 checks `unknown` and `other-general-jobs`, counts them for each country, and calculates `100 × missing / postings`. Q2 shows agreement and the hidden sectors from the completed Gemini run. Q3 shows before, after, and `after − before` for the top three sectors in each country. Every question has its own function.

In [ ]:
MISSING = ["unknown", "other-general-jobs"]
COUNTRIES = ["Germany", "Switzerland", "France", "Spain", "Netherlands", "Italy", "United Kingdom", "USA"]

def is_missing(tag):
    return tag in MISSING

def question_1(df):
    answer = []
    for country in COUNTRIES:
        tags = df[df["country"] == country]["category_tag"]
        missing = sum(tags.apply(is_missing))
        answer.append({"country": country, "postings": len(tags),
                       "missing": missing, "percent": round(100 * missing / len(tags), 1)})
    return pd.DataFrame(answer)

def question_2(df):
    total = df[df["part"] == "agreement"]
    matches = int(total["matches"][0])
    checked = int(total["checked"][0])
    print("Agreement:", matches, "of", checked, "=", round(100 * matches / checked, 1), "%")
    print("Missing categories filled:", int(total["filled"][0]), "of", int(total["missing"][0]))
    hidden = df[df["part"] == "hidden"]
    return pd.DataFrame({"sector": hidden["sector"], "count": hidden["count"].apply(int)})

def question_3(df):
    shifts = df[df["part"] == "shift"]
    return pd.DataFrame({"country": shifts["country"], "sector": shifts["sector"],
                         "before": shifts["before"].apply(int), "after": shifts["after"].apply(int),
                         "change": shifts["change"].apply(int)})

## Step 3 · Use Gemini on a missing job title

`JobChatbot` is the class: `__init__` stores the jobs table and `label_one` sends the first missing title to Gemini. The result is one **live illustration for Q2**. It does not overwrite or change the completed counts. If the secret or API is unavailable, the message explains why; Q1–Q3 can still run from the public files.

In [ ]:
LABELS = [
    "healthcare & nursing", "engineering", "trade & construction",
    "it & software development", "sales & business development",
    "accounting, banking & finance", "logistics, driver & supply chain",
    "teaching & education", "hospitality & catering",
    "administrative & office support", "hr & recruitment",
    "marketing, pr & advertising", "customer service & support",
    "social work & care", "maintenance, cleaning & facility", "other / specialized"
]

class JobChatbot:
    def __init__(self, df):
        self.jobs = df

    def label_one(self):
        for i in range(len(self.jobs)):
            if is_missing(self.jobs["category_tag"][i]):
                title = self.jobs["title"][i]
                break
        try:
            client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
            prompt = "Classify this job title. Choose only from: " + ", ".join(LABELS)
            prompt += "\nTitle: " + title + "\nReturn only the sector."
            response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)
            label = response.text.strip().lower()
            if label not in LABELS:
                label = "uncertain"
            print("Live Gemini example:", title, "→", label)
        except Exception as error:
            print("Live Gemini example unavailable:", error)

In [ ]:
chatbot = JobChatbot(jobs)
chatbot.label_one()

Live Gemini example: Teaching Assistant → teaching & education


In [ ]:
class JobChatbot:
    def __init__(self, df, completed_df=None):
        self.jobs = df
        self.completed = completed_df
        self.client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

    def label_one(self):
        for i in range(len(self.jobs)):
            if is_missing(self.jobs["category_tag"][i]):
                title = self.jobs["title"][i]
                break
        try:
            prompt = "Classify this job title. Choose only from: " + ", ".join(LABELS)
            prompt += "\nTitle: " + title + "\nReturn only the sector."
            response = self.client.models.generate_content(
                model="gemini-3.5-flash-lite", contents=prompt
            )
            label = response.text.strip().lower()
            if label not in LABELS:
                label = "uncertain"
            print("Live Gemini example:", title, "→", label)
        except Exception as error:
            print("Live Gemini example unavailable:", error)

    def ask(self, question):
        # Build a concise text summary of the dataset tables for Gemini to reference
        context = f"""
        Dataset overview: {len(self.jobs)} total job postings across {COUNTRIES}.

        Q1 - Missing Categories by Country:
        {question_1(self.jobs).to_string(index=False)}
        """
        if self.completed is not None:
            context += f"""
            Completed Gemini Classification Results:
            {self.completed.to_string(index=False)}
            """

        prompt = (
            f"You are a data analyst assistant. Answer the user's question using ONLY "
            f"the dataset summary provided below.\n\n{context}\n\nQuestion: {question}"
        )
        try:
            response = self.client.models.generate_content(
                model="gemini-3.5-flash-lite", contents=prompt
            )
            print(response.text.strip())
        except Exception as error:
            print("Could not answer question:", error)

## Step 4 · See the answers and decision

Q1 is calculated from the original 4,000 rows. Q2 and Q3 display saved aggregate answers, previously produced by Gemini and pandas from all 4,000 titles. They are counts of sampled postings after relabeling, **not changes in employment**. Agreement with source tags is not verified accuracy.

In [ ]:
print("Q1 · Missing categories by country")
print(question_1(jobs))
print("\nQ2 · Agreement and hidden sectors")
print(question_2(completed))
print("\nQ3 · Top sectors after filling missing categories")
print(question_3(completed))
print("\nDecision: Repair categories in Germany first; test engineering campaigns there.")

Q1 · Missing categories by country
          country  postings  missing  percent
0         Germany       500      434     86.8
1     Switzerland       500      355     71.0
2          France       500      325     65.0
3           Spain       500      323     64.6
4     Netherlands       500      281     56.2
5           Italy       500      159     31.8
6  United Kingdom       500        2      0.4
7             USA       500        0      0.0

Q2 · Agreement and hidden sectors
Agreement: 1355 of 2091 = 64.8 %
Missing categories filled: 1876 of 1879
                              sector  count
1               trade & construction    407
2   logistics, driver & supply chain    227
3                        engineering    204
4      accounting, banking & finance    176
5       sales & business development    157
6               healthcare & nursing    132
7   maintenance, cleaning & facility    122
8          it & software development    102
9                other / specialized     97
10 

In [ ]:
chatbot = JobChatbot(jobs, completed)

In [ ]:
chatbot.ask("What is the country with the most missing categories?")

Based on the provided dataset summary, the country with the most missing categories is **Germany**, with 434 missing postings (86.8%).
